<a href="https://colab.research.google.com/github/chhaviluthra08/Myntra-CatMan-Intelligence/blob/main/myntra.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
# ==============================================================================
# PROJECT: MYNTRA CATEGORY INTELLIGENCE & SELLER HEALTH ANALYSIS
# DATASET SCALE: 1.06 MILLION SKUs
# ==============================================================================

import kagglehub
import pandas as pd
import os

# --- DATA ACQUISITION & INITIALIZATION ---
# Utilizing KaggleHub for high-speed cloud data retrieval
path = kagglehub.dataset_download("ronakbokaria/myntra-products-dataset")

# Identifying and loading primary relational data from the source directory
files = os.listdir(path)
csv_file = [f for f in files if f.endswith('.csv')][0]
df = pd.read_csv(os.path.join(path, csv_file))

print(f"Pipeline initialized. Total catalog size: {len(df):,} products.")

# --- MODULE 1: CONSUMER DEMAND & ANCHOR PRODUCT IDENTIFICATION ---
# Objective: Define 'Anchor Products' by calculating a weighted Popularity Score.
# This metric balances qualitative (Rating) and quantitative (RatingTotal) social proof.
df['popularity_score'] = df['rating'] * df['ratingTotal']
top_anchors = df.sort_values(by='popularity_score', ascending=False).head(10)

# --- MODULE 2: REVENUE & MARGIN OPTIMIZATION ---
# Objective: Quantify discount depth to analyze margin erosion across the catalog.
# Formula: (MRP - Selling Price) / MRP, represented as a percentage.
df['discount_percentage'] = ((df['mrp'] - df['price']) / df['mrp']) * 100

# --- MODULE 3: STRATEGIC PARTNER (SELLER) PERFORMANCE AUDIT ---
# Objective: Aggregate performance metrics by seller to evaluate 'Brand Health.'
# Key KPI focus: Volume vs. Customer Satisfaction (Mean Rating) vs. Pricing Aggression.
seller_health = df.groupby('seller').agg({
    'rating': 'mean',
    'id': 'count',
    'discount_percentage': 'mean'
}).rename(columns={'id': 'product_count'}).sort_values(by='product_count', ascending=False)

print("\n--- EXECUTIVE SUMMARY: SELLER PERFORMANCE AUDIT ---")
print(seller_health.head(10))

# --- MODULE 4: SELECTION GAP & OPPORTUNITY ANALYSIS ---
# Objective: Segment the catalog into a Category Proxy to identify under-served growth drivers.
df['category_proxy'] = df['name'].str.split().str[0]

category_strategy = df.groupby('category_proxy').agg({
    'rating': 'mean',
    'discount_percentage': 'mean',
    'id': 'count'
}).rename(columns={'id': 'sku_count'})

# Filtering for 'High-Potential' segments: Categories with high ratings but low relative SKU density.
selection_gaps = category_strategy[category_strategy['sku_count'] > 100].sort_values(by='rating', ascending=False)

print("\n--- STRATEGIC GROWTH OPPORTUNITIES: CATEGORY SELECTION GAPS ---")
print(selection_gaps.head(10))

Using Colab cache for faster access to the 'myntra-products-dataset' dataset.
Dataset loaded with 1060213 products.
Index(['id', 'name', 'img', 'asin', 'price', 'mrp', 'rating', 'ratingTotal',
       'discount', 'seller', 'purl'],
      dtype='object')
--- SELLER HEALTH REPORT (Top 10 by Volume) ---
                         rating  product_count  discount_percentage
seller                                                             
Roadster               3.276359          10651            53.192729
H&M                    1.552902           6667             6.538100
Puma                   2.720961           6579            37.880824
max                    1.553731           6486             1.202160
Anouk                  1.167571           6158            63.771402
KALINI                 0.362437           5793            69.353504
JC Collection          0.195880           5631            33.185873
HRX by Hrithik Roshan  3.255857           5575            49.483721
Friskers           